# SemEval-2026 Task 13 - Task C (Kaggle Multi-GPU Version)\nSetup to utilize 2x T4 GPUs, cleanly retrieve Kaggle Secrets for Auth, W&B logging, HuggingFace Hub pushing, and checkpoint resuming for 4-CLASS Hybrid Authorship Detection.

In [ ]:
# ==========================================
# CONFIGURATION VARIABLES
# ==========================================
# Fetch Authentication Tokens safely from Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    WANDB_TOKEN = user_secrets.get_secret("WANDB_TOKEN")
    HF_REPO_ID = user_secrets.get_secret("HF_REPO_ID")
    print("✅ Loaded HF_TOKEN and WANDB_TOKEN from Kaggle Secrets.")
except ImportError:
    import os
    print("⚠️ Not running on Kaggle. Falling back to environment variables.")
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    WANDB_TOKEN = os.environ.get("WANDB_TOKEN", "")
    HF_REPO_ID = os.environ.get("HF_REPO_ID", "")

# Note: Make sure to attach your secrets in the Kaggle Notebook via: Add-ons -> Secrets

HF_REPO_ID = "sohailtsm/semeval_taskC_model" # Repository to push the model to
WANDB_PROJECT = "semeval-2026-task13"

# Data Limits (Set to None to use the full dataset)
MAX_TRAIN_SAMPLES = None    
MAX_VAL_SAMPLES = None

# Model & Training Hyperparameters
MODEL_NAME = "microsoft/codebert-base"
MAX_LENGTH = 256
BATCH_SIZE = 32      # Per-device batch size (Effective batch size = 32 * 2 GPUs = 64)
GRADIENT_ACCUMULATION_STEPS = 1 # Number of steps to accumulate gradients (Increase if you lower BATCH_SIZE)
NUM_EPOCHS = 5
EVAL_STEPS = 200     # Evaluate every 200 steps (multiple times per epoch)
LEARNING_RATE = 1e-5
FP16_ENABLED = True  # Enable mixed precision for faster training on T4 GPUs

# Kaggle I/O Paths
OUTPUT_DIR = "/kaggle/working/task_c_model"
PREDICTIONS_FILE = "/kaggle/working/evaluation_predictions.csv"
METRICS_FILE = "/kaggle/working/evaluation_metrics.csv"
MISCLASSIFIED_FILE = "/kaggle/working/misclassified_samples.csv"
TEST_MISCLASSIFIED_FILE = "/kaggle/working/test_misclassified_samples.csv"
SUBMISSION_FILE = "/kaggle/working/submission.csv"


## Step 1: Install Dependencies

In [ ]:
# Install required packages
# Avoid using --upgrade for everything to prevent breaking Kaggle's pre-installed PyTorch/TorchAO environment
!pip install transformers==4.45.0 datasets huggingface_hub accelerate wandb -q

## Step 2: Import Libraries & Authenticate

In [ ]:
import os
import torch
import numpy as np
# PyTorch 2.6+ Checkpoint Resume Fix (Trainer compat)
_original_torch_load = torch.load
def safe_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)
torch.load = safe_torch_load
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
import wandb
from huggingface_hub import login
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, f1_score, precision_score, recall_score, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

# Authenticate with Weights & Biases
if WANDB_TOKEN:
    wandb.login(key=WANDB_TOKEN)
    os.environ["WANDB_PROJECT"] = WANDB_PROJECT
else:
    os.environ["WANDB_DISABLED"] = "true"
    print("WANDB_TOKEN not found. Weights & Biases logging is disabled.")

# Authenticate with HuggingFace Hub
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print("HF_TOKEN not found. Hugging Face Hub pushing is disabled.")

## Step 3: Define Training Class

In [ ]:
class CodeBERTTrainer:
    def __init__(self, max_length=MAX_LENGTH, model_name=MODEL_NAME):
        self.max_length = max_length
        self.model_name = model_name
        self.tokenizer = None
        self.model = None
        self.num_labels = None
        
    def load_and_prepare_data(self):
        """Load Task C data from HuggingFace"""
        try:
            print("Loading dataset from HuggingFace...")
            dataset = load_dataset("sohailtsm/my-semeval-dataset", "C")
            
            df = dataset['train'].to_pandas()
            val_df = dataset['validation'].to_pandas()
            
            df = df.dropna(subset=['code', 'label'])
            val_df = val_df.dropna(subset=['code', 'label'])
            
            df['label'] = df['label'].astype(int)
            val_df['label'] = val_df['label'].astype(int)
            self.num_labels = df['label'].nunique()
            
            print(f"\nTotal Train samples: {len(df):,}, Total Validation samples: {len(val_df):,}")
            
            # Apply data usage limits if defined
            if MAX_TRAIN_SAMPLES:
                df = df.sample(n=min(MAX_TRAIN_SAMPLES, len(df)), random_state=42).reset_index(drop=True)
            if MAX_VAL_SAMPLES:
                val_df = val_df.sample(n=min(MAX_VAL_SAMPLES, len(val_df)), random_state=42).reset_index(drop=True)
                
            print(f"Using - Train: {len(df):,}, Val: {len(val_df):,}")
            
            return df, val_df
            
        except Exception as e:
            print(f"Error loading dataset: {e}")
            raise
    
    def initialize_model_and_tokenizer(self):
        """Initialize CodeBERT model and tokenizer"""
        print(f"\nInitializing {self.model_name} model and tokenizer...")
        
        self.tokenizer = RobertaTokenizer.from_pretrained(self.model_name)
        self.model = RobertaForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=self.num_labels,
            problem_type="single_label_classification"
        )
        print(f"Model initialized with {self.num_labels} labels.")
    
    def tokenize_function(self, examples):
        """Tokenize code snippets"""
        return self.tokenizer(
            examples['code'],
            truncation=True,
            padding=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
    
    def prepare_datasets(self, train_df, val_df):
        """Convert DataFrames to HuggingFace Datasets and tokenize"""
        print("\nPreparing datasets for training...")
        
        train_dataset = Dataset.from_pandas(train_df[['code', 'label']])
        val_dataset = Dataset.from_pandas(val_df[['code', 'label']])
        
        train_dataset = train_dataset.map(self.tokenize_function, batched=True, remove_columns=['code'])
        val_dataset = val_dataset.map(self.tokenize_function, batched=True, remove_columns=['code'])
        
        train_dataset = train_dataset.rename_column('label', 'labels')
        val_dataset = val_dataset.rename_column('label', 'labels')
        
        return train_dataset, val_dataset
    
    def compute_metrics(self, eval_pred):
        """Compute evaluation metrics"""
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        
        accuracy = accuracy_score(labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro')
        
        return {
            'accuracy': accuracy,
            'f1': f1,
            'precision': precision,
            'recall': recall
        }
    
    def train(self, train_dataset, val_dataset):
        """Train the model"""
        push_to_hub = bool(HF_TOKEN)
        report_to = "wandb" if WANDB_TOKEN else "none"

        training_args = TrainingArguments(
            output_dir=OUTPUT_DIR,
            num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
            weight_decay=0.05,
            logging_dir='./logs',
            logging_steps=50,
            evaluation_strategy="steps",  # Evaluate every EVAL_STEPS
            eval_steps=EVAL_STEPS,
            save_strategy="steps",        # Save every EVAL_STEPS
            save_steps=EVAL_STEPS,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss", # Changed from f1 to stop when loss diverges
            greater_is_better=False,           # Lower loss is better
            label_smoothing_factor=0.1,        # Prevents overconfidence directly!
            learning_rate=LEARNING_RATE,
            lr_scheduler_type="cosine",
            warmup_ratio=0.1,             # Warm up for the first 10%
            save_total_limit=2,           # Keep only the last 2 best checkpoints
            report_to=report_to,
            fp16=FP16_ENABLED,            # Mixed precision training
            push_to_hub=push_to_hub,
            hub_model_id=HF_REPO_ID if push_to_hub else None,
            hub_strategy="checkpoint",    # Push to hub at every save
            hub_token=HF_TOKEN if push_to_hub else None
        )
        
        data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)
        
        from transformers.trainer_utils import get_last_checkpoint
        import os
        last_checkpoint = None
        if os.path.isdir(OUTPUT_DIR):
            last_checkpoint = get_last_checkpoint(OUTPUT_DIR)
            
        if last_checkpoint is None and push_to_hub:
            from huggingface_hub import HfApi, snapshot_download
            try:
                print(f"\n🔍 Local dir wiped. Searching HuggingFace Hub ({HF_REPO_ID}) for checkpoints...")
                HfApi(token=HF_TOKEN).model_info(HF_REPO_ID)
                
                # Fetch BOTH standard and custom 'last-checkpoint' folder patterns
                dl_dir = snapshot_download(repo_id=HF_REPO_ID, token=HF_TOKEN, allow_patterns=["checkpoint-*/*", "last-checkpoint/*"])
                
                hf_checkpoint = get_last_checkpoint(dl_dir)
                if not hf_checkpoint:
                    if os.path.exists(os.path.join(dl_dir, "last-checkpoint")):
                        hf_checkpoint = os.path.join(dl_dir, "last-checkpoint")
                        
                if hf_checkpoint:
                    last_checkpoint = hf_checkpoint
                    print(f"✅ Successfully pulled remote Hub checkpoint state from {last_checkpoint}")
            except Exception as e:
                print(f"⚠️ Hub resume failed: {e}")
                
        print("\nStarting training...")
        if last_checkpoint is not None:
             print(f"✅ Resuming training from precisely: {last_checkpoint}...")

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            tokenizer=self.tokenizer,
            data_collator=data_collator,
            compute_metrics=self.compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
        )
        
        trainer.train(resume_from_checkpoint=last_checkpoint)
        
        # Save final best model locally
        trainer.save_model(OUTPUT_DIR)
        self.tokenizer.save_pretrained(OUTPUT_DIR)
        
        # Final push to hub if enabled
        if push_to_hub:
            trainer.push_to_hub("End of training")
            
        print(f"\n✅ Training completed. Best model saved to {OUTPUT_DIR}")
        
        return trainer

## Step 4: Data Preparation

In [ ]:
# Execute Data Preparation and Selection
trainer_mgr = CodeBERTTrainer()
train_df, val_df = trainer_mgr.load_and_prepare_data()

# Prepare datasets
trainer_mgr.initialize_model_and_tokenizer()
train_dataset, val_dataset = trainer_mgr.prepare_datasets(train_df, val_df)

## Step 5: Execute Training\n*Note: Running this will automatically resume from the latest checkpoint if stopped prematurely.*

In [ ]:
# Execute Training
# If you stop execution here and run again, it will resume from the last saved epoch checkpoint.
trained_model = trainer_mgr.train(train_dataset, val_dataset)

## Step 6: Evaluate Validation Set\n*Note: Evaluates validation misclassifications. Disables W&B logging strictly to prevent crashes if training session was closed.*

In [ ]:
# Evaluate and Output Misclassifications
import os
from huggingface_hub import HfApi

# If training was skipped or notebook restarted, we need to load the model
if 'trained_model' not in locals():
    print("Training phase bypassed. Attempting to locate the best model for evaluation...")
    
    evaluate_model_path = MODEL_NAME # Default to base model
    
    # 1. Check local directory for saved model
    if os.path.exists(os.path.join(OUTPUT_DIR, "config.json")):
        print(f"✅ Found local trained model in {OUTPUT_DIR}")
        evaluate_model_path = OUTPUT_DIR
    # 2. Check HuggingFace Hub
    elif HF_REPO_ID and HF_REPO_ID != "your_username/semeval_taskA_model":
        try:
            HfApi().model_info(HF_REPO_ID)
            print(f"✅ Found model on HuggingFace Hub: {HF_REPO_ID}")
            evaluate_model_path = HF_REPO_ID
        except Exception:
            print("⚠️ Model not found on hub, falling back to base model.")
    
    # Initialize the model using the determined path
    trainer_mgr.model_name = evaluate_model_path
    trainer_mgr.initialize_model_and_tokenizer()
    
    # Recreate a standalone trainer for evaluation without w&b logging
    from transformers import TrainingArguments
    eval_args = TrainingArguments(output_dir="/tmp/eval", per_device_eval_batch_size=BATCH_SIZE, report_to="none")
    trained_model = Trainer(
        model=trainer_mgr.model,
        args=eval_args,
        tokenizer=trainer_mgr.tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=trainer_mgr.tokenizer)
    )
else:
    # If trained_model exists from training, remove the WandbCallback
    # This ensures predict() doesn't crash if the W&B run was previously finished.
    from transformers.integrations import WandbCallback
    try:
        trained_model.remove_callback(WandbCallback)
    except ValueError:
        pass # Callback wasn't present

print("\n📊 Predicting on validation dataset...")
predictions = trained_model.predict(val_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

# Extract original code to store alongside misclassifications
codes = val_df['code'].tolist()

LABEL_MAP = {
    0: 'human', 1: 'machine', 2: 'hybrid', 3: 'adversarial'
}

eval_results_df = pd.DataFrame({
    'sample_id': range(len(y_true)),
    'code': codes,
    'true_label': y_true,
    'predicted_label': y_pred,
    'correct': y_true == y_pred,
    'true_label_name': [LABEL_MAP.get(x, str(x)) for x in y_true],
    'predicted_label_name': [LABEL_MAP.get(x, str(x)) for x in y_pred]
})

# Save detailed predictions
eval_results_df.to_csv(PREDICTIONS_FILE, index=False)
print(f"✅ Detailed predictions saved to: {PREDICTIONS_FILE}")

# Identify and save misclassifications
misclassified_df = eval_results_df[~eval_results_df['correct']]
misclassified_df.to_csv(MISCLASSIFIED_FILE, index=False)
print(f"🔍 Found {len(misclassified_df)} misclassified samples.")
print(f"✅ Misclassified samples saved to: {MISCLASSIFIED_FILE}")

# Calculate metrics
accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro')
print("\n" + "="*60)
print("📊 EVALUATION RESULTS")
print("="*60)
print(f"Accuracy:        {accuracy:.4f}")
print(f"Macro F1:        {macro_f1:.4f} ⭐")
print("\n📋 Detailed Classification Report:")
unique_labels = sorted(list(set(y_true)))
target_names = [LABEL_MAP.get(l, str(l)) for l in unique_labels]
print(classification_report(y_true, y_pred, labels=unique_labels, target_names=target_names, digits=4))

# Keep minimal metrics file
results_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Macro F1'],
    'Value': [accuracy, macro_f1]
})
results_df.to_csv(METRICS_FILE, index=False)
print(f"✅ Metrics saved to: {METRICS_FILE}")

# Ensure W&B run is finished properly after everything is done
if 'wandb' in globals() and wandb.run is not None:
    wandb.finish()


## Step 7: Predict on Public Test Set & Create Submission\n*Note: Generates the `submission.csv` for the leaderboard.*

In [ ]:
# Step 7: Predict on Test Set & Create Submission File
print("\n🚀 Loading Test Set from HuggingFace...")
test_dataset_raw = load_dataset("DaniilOr/SemEval-2026-Task13", "C", split="test")
test_df = test_dataset_raw.to_pandas()

# Test sets might not have labels publicly; we only need 'code' and 'id'
if 'label' not in test_df.columns:
    test_df['label'] = 0 # Dummy label for Trainer

print(f"Total Test samples: {len(test_df):,}")

print("\nPreparing test dataset...")
test_dataset = Dataset.from_pandas(test_df[['code', 'label']])
test_dataset = test_dataset.map(trainer_mgr.tokenize_function, batched=True, remove_columns=['code'])
test_dataset = test_dataset.rename_column('label', 'labels')

print("\n📊 Predicting on test dataset...")
test_predictions = trained_model.predict(test_dataset)
test_y_pred = np.argmax(test_predictions.predictions, axis=1)

# Create formal submission file
if 'id' in test_df.columns:
    submission_df = pd.DataFrame({
        'id': test_df['id'],
        'label': test_y_pred
    })
else:
    submission_df = pd.DataFrame({
        'id': range(len(test_y_pred)),
        'label': test_y_pred
    })

submission_df.to_csv(SUBMISSION_FILE, index=False)
print(f"✅ Formal Submission file saved to: {SUBMISSION_FILE}")

# Calculate metrics if test set actually contains real public labels
if 'label' in test_df.columns and test_df['label'].nunique() > 1:
    test_y_true = test_predictions.label_ids
    accuracy = accuracy_score(test_y_true, test_y_pred)
    macro_f1 = f1_score(test_y_true, test_y_pred, average='macro')
    print("\n" + "="*60)
    print("📊 PUBLIC TEST SET METRICS")
    print("="*60)
    print(f"Test Accuracy:   {accuracy:.4f}")
    print(f"Test Macro F1:   {macro_f1:.4f} ⭐")
    
    # Save test misclassifications
    codes = test_df['code'].tolist()
    LABEL_MAP = {
        0: 'human', 1: 'machine', 2: 'hybrid', 3: 'adversarial'
    }
    test_results_df = pd.DataFrame({
        'sample_id': test_df['id'] if 'id' in test_df.columns else range(len(test_y_true)),
        'code': codes,
        'true_label': test_y_true,
        'predicted_label': test_y_pred,
        'correct': test_y_true == test_y_pred,
        'true_label_name': [LABEL_MAP.get(x, str(x)) for x in test_y_true],
        'predicted_label_name': [LABEL_MAP.get(x, str(x)) for x in test_y_pred]
    })
    
    test_misclass_df = test_results_df[~test_results_df['correct']]
    test_misclass_df.to_csv(TEST_MISCLASSIFIED_FILE, index=False)
    print(f"🔍 Found {len(test_misclass_df)} test misclassified samples.")
    print(f"✅ Test misclassified samples saved to: {TEST_MISCLASSIFIED_FILE}")
